In [ ]:
from pyspark.sql import functions as F

from src.common.silver_incremental import (
    NO_TABLES_SENTINEL,
    append_silver_audit,
    build_audit_record,
    display_step_summary,
    ensure_silver_audit_table,
    filter_pending_step_rows,
    list_processing_tables,
    merge_silver_table,
    resolve_merge_keys,
    silver_audit_path,
    silver_audit_table,
    summary_from_record,
)

dbutils.widgets.text('environment', 'dev')
dbutils.widgets.text('catalog_name', 'naturapet_dev')
dbutils.widgets.text('storage_account', 'demodldb')
dbutils.widgets.text('storage_container', 'democodex')
dbutils.widgets.text('data_year', '2026')
dbutils.widgets.text('domain_name', 'comercial')

environment = dbutils.widgets.get('environment').strip().lower()
catalog = dbutils.widgets.get('catalog_name').strip()
storage_account = dbutils.widgets.get('storage_account').strip()
storage_container = dbutils.widgets.get('storage_container').strip()
data_year = dbutils.widgets.get('data_year').strip()
domain = dbutils.widgets.get('domain_name').strip().lower()
silver_schema = f'{domain}_silver'
storage_uri = f'abfss://{storage_container}@{storage_account}.dfs.core.windows.net'
silver_root = f'{storage_uri}/external/{environment}/{domain}/silver/{data_year}'

spark.sql(f'CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{silver_schema}`')

STEP_NAME = 'null_corrections'
PREVIOUS_STEP = 'schema_standardization'
audit_table = silver_audit_table(catalog, silver_schema)
audit_path = silver_audit_path(silver_root)
ensure_silver_audit_table(spark, audit_table, audit_path)

def silver_tables():
    return [row.tableName for row in spark.sql(f'SHOW TABLES IN `{catalog}`.`{silver_schema}`').collect() if not row.isTemporary]

def normalize_empty_strings(df):
    for field in df.schema.fields:
        if field.dataType.simpleString() == 'string':
            cleaned = F.trim(F.col(field.name))
            df = df.withColumn(field.name, F.when(cleaned == '', F.lit(None)).otherwise(cleaned))
    return df

def apply_null_policy(table_name, df):
    df = normalize_empty_strings(df)
    if table_name == 'dim_fecha' and {'es_festivo_colombia', 'festivo_nombre'}.issubset(df.columns):
        df = df.withColumn('festivo_nombre', F.when(F.col('es_festivo_colombia') == F.lit(True), F.col('festivo_nombre')).otherwise(F.lit(None)))
    if table_name == 'dim_empleado' and 'tienda_id' in df.columns:
        df = (df.withColumn('es_empleado_corporativo', F.col('tienda_id').isNull())
                .withColumn('tienda_id_modelo', F.coalesce(F.col('tienda_id'), F.lit('NO_APLICA')))
              )
    if table_name == 'fact_logistica_entregas':
        if 'empleado_repartidor_id' in df.columns:
            df = df.withColumn(
                'empleado_repartidor_null_reason',
                F.when(F.col('empleado_repartidor_id').isNull() & F.col('transportista_id').isNotNull(), F.lit('transportista_tercero'))
                 .when(F.col('empleado_repartidor_id').isNull(), F.lit('sin_repartidor_informado'))
                 .otherwise(F.lit(None))
            )
        if 'fecha_entrega' in df.columns:
            df = df.withColumn(
                'fecha_entrega_null_reason',
                F.when((F.col('fecha_entrega').isNull()) & (F.col('estado_entrega') == F.lit('Cancelado en ruta')), F.lit('cancelado_en_ruta'))
                 .when(F.col('fecha_entrega').isNull(), F.lit('pendiente_o_no_entregado'))
                 .otherwise(F.lit(None))
            )
    if table_name == 'fact_ventas_cabecera' and 'campana_id' in df.columns:
        df = df.withColumn('campana_null_reason', F.when(F.col('campana_id').isNull(), F.lit('venta_sin_campana')).otherwise(F.lit(None)))
    return df.withColumn('_np_silver_step', F.lit('null_corrections')).withColumn('_np_silver_processed_ts', F.current_timestamp())

def write_silver(df, table_name):
    target_table = f'`{catalog}`.`{silver_schema}`.`{table_name}`'
    target_path = f'{silver_root}/{table_name}'
    df = df.withColumn('_np_silver_step', F.lit(STEP_NAME)).withColumn('_np_silver_processed_ts', F.current_timestamp())
    merge_keys = resolve_merge_keys(table_name, df.columns)
    merge_silver_table(spark, df, target_table, target_path, merge_keys)
    return {'target_table': target_table, 'target_path': target_path, 'rows': df.count()}

summary = []
tables = list_processing_tables(spark, catalog, silver_schema)
if not tables:
    record = build_audit_record(
        environment=environment,
        domain_name=domain,
        step_name=STEP_NAME,
        table_name=NO_TABLES_SENTINEL,
        source_table=f'`{catalog}`.`{silver_schema}`',
        target_table=f'`{catalog}`.`{silver_schema}`',
        input_rows=0,
        output_rows=0,
        status='NO_DATA',
        message='No existen tablas Silver para procesar en este dominio.',
        target_path=silver_root,
    )
    append_silver_audit(spark, audit_table, audit_path, record)
    summary.append(summary_from_record(record))

for table_name in tables:
    target_table = f'`{catalog}`.`{silver_schema}`.`{table_name}`'
    target_path = f'{silver_root}/{table_name}'
    df = filter_pending_step_rows(spark.table(target_table), PREVIOUS_STEP)
    input_rows = df.count()

    if input_rows == 0:
        record = build_audit_record(
            environment=environment,
            domain_name=domain,
            step_name=STEP_NAME,
            table_name=table_name,
            source_table=target_table,
            target_table=target_table,
            input_rows=0,
            output_rows=0,
            status='NO_DATA',
            message=f'No hay filas pendientes desde {PREVIOUS_STEP}.',
            target_path=target_path,
        )
        append_silver_audit(spark, audit_table, audit_path, record)
        summary.append(summary_from_record(record))
        continue

    df = apply_null_policy(table_name, df)
    write_result = write_silver(df, table_name)
    record = build_audit_record(
        environment=environment,
        domain_name=domain,
        step_name=STEP_NAME,
        table_name=table_name,
        source_table=target_table,
        target_table=target_table,
        input_rows=input_rows,
        output_rows=write_result['rows'],
        status='SUCCESS',
        message=f'Filas pendientes desde {PREVIOUS_STEP} procesadas incrementalmente.',
        target_path=target_path,
    )
    append_silver_audit(spark, audit_table, audit_path, record)
    summary.append(summary_from_record(record))

display(display_step_summary(spark, summary))
